# 1 · Quickstart

This notebook loads and **validates** a `data4simplace` configuration,
inspects the enabled stages, and shows the same *dry-run* plan the CLI
prints. It needs no external datasets.

## Import and check the version

In [1]:
import data4simplace

print('data4simplace', data4simplace.__version__)

data4simplace 0.1.0


## Write a minimal configuration

We create a small config in a temporary directory so the notebook is
self-contained. In practice you would edit the project `config.yaml`.

In [ ]:
import tempfile, textwrap, os
from pathlib import Path

workdir = Path(tempfile.mkdtemp(prefix='d4s_'))
config_yaml = workdir / 'config.yaml'
config_yaml.write_text(textwrap.dedent('''
    flags:
      run_climate_processing: true
      run_soil_processing: true
      compute_ptf: true
      run_npk_processing: false
      apply_agricultural_mask: false
      export_simplace_weather: true
      export_simplace_soil: true
      export_simplace_management: false
    grid:
      resolution_deg: 0.1
      min_lon: 11.2
      max_lon: 14.8
      min_lat: 51.3
      max_lat: 53.6
    time:
      start: "1979-01-01"
      end: "1979-01-31"
    paths:
      mswx_root: "/data01/FDS/muduchuru/Atmos/MSWX"
      output_dir: "%s"
''' % (workdir / 'output')).strip())
print(config_yaml.read_text())

## Load and validate it

[`load_config`](../api/config.md#data4simplace.config.load_config) parses the YAML and returns a validated,
 immutable [`PipelineConfig`](../api/config.md#data4simplace.config.PipelineConfig).
 Unknown keys or out-of-range bounds raise immediately.

In [ ]:
from data4simplace import load_config

config = load_config(config_yaml)
type(config)

In [ ]:
# The grid section, fully typed
config.grid

## Inspect the execution plan (same as `--dry-run`)

In [ ]:
enabled = [name for name, on in config.flags.model_dump().items() if on]
print('Enabled stages:')
for stage in enabled:
    print(' -', stage)

## Validation catches mistakes early

A bad bounding box (min ≥ max) fails at load time rather than deep inside
a processing stage.

In [ ]:
from pydantic import ValidationError

bad = config_yaml.read_text().replace('max_lon: 14.8', 'max_lon: 10.0')
bad_path = workdir / 'bad.yaml'
bad_path.write_text(bad)

try:
    load_config(bad_path)
except (ValidationError, ValueError) as exc:
    print('Rejected as expected:')
    print(exc)

## Next

- [The 10 km target grid](02_target_grid.ipynb)
- [Pedotransfer functions](03_pedotransfer_functions.ipynb)
- [Running the full pipeline](04_full_pipeline.ipynb)